In [ ]:
import matplotlib.pyplot as plt

import xarray as xr
import numpy as np

In [ ]:

%load_ext autoreload
%autoreload 2

In [ ]:
aero_tot = 'aerosol-total'
aero_cld = 'aerosol-cloud_interactions'
aero_rad = 'aerosol-radiation_interactions'
bc_on_snow = 'bc_on_snow'
aero_tot_wbc = 'aerosol-total-with_bc-snow'

In [ ]:
from ar6_ch6_rcmipfigs.constants import OUTPUT_DATA_DIR

PATH_DT_INPUT = OUTPUT_DATA_DIR / 'fig6_22_and_6_24'/'dT_data_RCMIP_recommendation.nc'
PATH_DT_INPUT_minor = OUTPUT_DATA_DIR / 'fig6_22_and_6_24' / 'dT_data_RCMIP_recommendation_minor.nc'

PATH_UNCERT_DT_INPUT = OUTPUT_DATA_DIR /'fig6_22_and_6_24' /  'dT_uncertainty_data_FaIR_chris.nc'
PATH_UNCERT_DT_OUTPUT = OUTPUT_DATA_DIR / 'fig6_22_and_6_24' / 'dT_uncertainty_data_FaIR_chris_ed02-02-02.nc'

In [ ]:
ds_unc = xr.open_dataset(PATH_UNCERT_DT_INPUT)

In [ ]:
ds_unc

In [ ]:
v_sigma2_bot = 'p16-p50'
v_sigma2_top = 'p84-p50'
np.abs(ds_unc[v_sigma2_bot]).sel(scenario='ssp119', variable=bc_on_snow).plot(label='5%')
np.abs(ds_unc[v_sigma2_top]).sel(scenario='ssp119', variable=bc_on_snow).plot(label='95%')
plt.legend()
plt.show()

v_sigma2_bot = 'p16'
v_sigma2_top = 'p84'
v_med = 'p50'
ds_unc[v_sigma2_bot].sel(scenario='ssp119', variable=bc_on_snow).plot(label='5%')
ds_unc[v_sigma2_top].sel(scenario='ssp119', variable=bc_on_snow).plot(label='95%')
ds_unc[v_med].sel(scenario='ssp119', variable=bc_on_snow).plot(label='median')
plt.legend()
plt.show()

In [ ]:
p05 = 'p05'
p95 = 'p95'
p16 = 'p16'

p84 = 'p84'
median = 'p50'
es_sig = 'estimated_sigma'
es_median = 'estimated_median'
ds_unc[es_sig] = np.abs((ds_unc[p16] - ds_unc[p84]) / 2)

In [ ]:
ds_bc_snow = ds_unc.sel(variable=bc_on_snow)
ds_aero_tot = ds_unc.sel(variable=aero_tot)
sigma_sum = np.sqrt(ds_bc_snow[es_sig] ** 2 + ds_aero_tot[es_sig] ** 2)
median_sum = ds_bc_snow[median] + ds_aero_tot[median]
_coords = ds_bc_snow.coords
_coords['variable'] = [aero_tot_wbc]
ds_sum = xr.Dataset(coords=_coords)  # {'variable':[aero_tot_wbc]})
# sig_bc_snow.coords)
ds_sum

In [ ]:
from scipy.stats import norm

di = norm(loc=median_sum, scale=sigma_sum)

ds_sum[p05] = xr.DataArray(di.ppf(0.05), coords=ds_bc_snow[p05].coords)
ds_sum[p95] = xr.DataArray(di.ppf(0.95), coords=ds_bc_snow[p05].coords)
ds_sum[p16] = xr.DataArray(di.ppf(0.16), coords=ds_bc_snow[p05].coords)
ds_sum[p84] = xr.DataArray(di.ppf(0.84), coords=ds_bc_snow[p05].coords)
ds_sum[median] = xr.DataArray(di.ppf(0.50), coords=ds_bc_snow[p05].coords)

In [ ]:
for v in [p05, p16, p84, p95, median]:
    ds_sum[f'{v}-{median}'] = ds_sum[v] - ds_sum[median]

In [ ]:
cols = {p05: 'r', p95: 'b', median: 'g'}
for c in [p05, p95, median]:
    ds_bc_snow[c].sel(scenario='ssp119').plot(c=cols[c], label=c + ', bc', linestyle='dashed')
    ds_aero_tot[c].sel(scenario='ssp119').plot(c=cols[c], label=c + ', aero', linestyle='dotted')
    ds_sum[c].sel(scenario='ssp119').plot(c=cols[c], linestyle='solid', label=c + ', sum')

# ds_aero_tot[p95].sel(scenario='ssp119').plot()
# ds_aero_tot[median].sel(scenario='ssp119').plot()
plt.legend()
plt.show()

In [ ]:
# noinspection PyDeprecation
ds_unc_upd = xr.concat([ds_unc.drop([es_sig]), ds_sum], dim='variable')
ds_unc_upd.variable
ds_unc_upd.to_netcdf(PATH_UNCERT_DT_OUTPUT)

In [ ]:
# excluded_HFCs = ['HFC-23','HFC-125','HFC-143a','HFC-227ea','HFC-236fa']
excluded_HFCs = ['HFC-23', 'HFC-236fa']  # 'HFC-125','HFC-227ea','HFC-143a',

In [ ]:
ordered_lifetime_ls = [
    'HFC-152a',
    'HFC-32',
    'HFC-245fa',
    'HFC-365mfc',
    'HFC-134a',
    'HFC-43-10mee',
    'HFC-125',
    'HFC-227ea',
    'HFC-143a',
    'HFC-236fa',
    'HFC-23',
]

hfcs_tau = {
    'HFC-152a': 1.6,
    'HFC-32': 5.4,
    'HFC-245fa': 7.9,
    'HFC-365mfc': 8.9,
    'HFC-134a': 14.0,
    'HFC-43-10mee': 17.0,
    'HFC-125': 30.0,
    'HFC-227ea': 36.0,
    'HFC-143a': 51.0,
    'HFC-236fa': 213.0,
    'HFC-23': 228,
}

In [ ]:
name_deltaT = 'Delta T'

In [ ]:
ds_DT = xr.open_dataset(PATH_DT_INPUT)
ds_DT_minor = xr.open_dataset(PATH_DT_INPUT_minor)

# hfc Delta T
hfc_dT = ds_DT_minor[name_deltaT].sel(percentile='recommendation')

In [ ]:
hfc_dT

In [ ]:
ls = list(ds_DT_minor['variable'].values)
# chocose only those with HFC in them
vars_HFCs = [v for v in ls if 'HFC-' in v]

vars_HFCs

In [ ]:
final_HFC_vars = [hfc for hfc in vars_HFCs if hfc not in excluded_HFCs]

final_HFC_vars

In [ ]:
hfs_sum_exclusive = hfc_dT.sel(variable=final_HFC_vars).sum('variable')

In [ ]:
hfs_sum_all = hfc_dT.sel(variable=vars_HFCs).sum('variable')

In [ ]:
rat = hfs_sum_exclusive / hfs_sum_all

In [ ]:
for scn in hfs_sum_all['scenario'].values:
    rat.sel(scenario=scn, year=slice(2020, 2100)).plot(label=scn)
    # hfs_sum_all.sel(scenario=scn).plot(label=scn)
    # hfs_sum_exclusive.sel(scenario=scn).plot(label=scn)

plt.legend()
plt.show()

In [ ]:
for hf in ordered_lifetime_ls:
    hfc_dT.sel(variable=hf, scenario='ssp334', year=slice(2000, 2100)).plot(label=hf + ', %s' % hfcs_tau[hf])
plt.legend()
plt.show()

In [ ]:
_da = ds_unc_upd['p95-p50'].sel(variable='HFCs')
for scn in _da['scenario'].values:
    _da.sel(scenario=scn).plot()
plt.show()

In [ ]:
ds_unc_upd2 = ds_unc_upd.copy(deep=True)

In [ ]:
v = 'p05-p50'
_tmp = (ds_unc_upd2[v]).loc[{'variable': 'HFCs'}] * rat
ds_unc_upd2[v].loc[{'variable': 'HFCs'}] = _tmp

In [ ]:
v = 'p95-p50'
_tmp = (ds_unc_upd2[v]).loc[{'variable': 'HFCs'}] * rat
ds_unc_upd2[v].loc[{'variable': 'HFCs'}] = _tmp

In [ ]:
ds_unc_upd2[v].sel(scenario='ssp585', variable='HFCs').plot()

ds_unc_upd[v].sel(scenario='ssp585', variable='HFCs').plot()

In [ ]:
ds_unc_upd2.to_netcdf(PATH_UNCERT_DT_OUTPUT)

In [ ]:
print(PATH_UNCERT_DT_OUTPUT)